# Benchmark de anomalias (PostgreSQL)

Este notebook ejecuta benchmark y sensibilidad de detectores de anomalias, incluyendo candidatos basados en PCA.


In [ ]:
from pathlib import Path
import sys
import warnings

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

warnings.filterwarnings("ignore")

CWD = Path.cwd().resolve()
candidates = [
    CWD,
    CWD.parent,
    CWD / "03_modelado" / "proyecto_ml_experimentos",
]
ROOT = next((p for p in candidates if (p / "src" / "anomaly.py").exists()), None)
if ROOT is None:
    raise RuntimeError("No se encontro la raiz de proyecto_ml_experimentos (src/anomaly.py).")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

for m in ["src", "src.datasets_postgres", "src.anomaly"]:
    if m in sys.modules:
        del sys.modules[m]

from src.datasets_postgres import load_anomaly_dataset
from src.anomaly import benchmark_anomaly, benchmark_anomaly_sensitivity


In [ ]:
df = load_anomaly_dataset()
display(df.head())

display(
    Markdown(
        f'''
### Calidad del dataset
- Filas (agencias): **{len(df):,}**
- Agencias unicas: **{df['agencia'].nunique():,}**
- Features usadas: `ratio_devolucion`, `ratio_rentabilidad`, `ratio_costo`, `ticket_promedio`
'''
    )
)


In [ ]:
res = benchmark_anomaly(df, contamination=0.10)
sens = benchmark_anomaly_sensitivity(df, contamination_grid=(0.05, 0.10, 0.15))

models_dir = ROOT / "models"
models_dir.mkdir(exist_ok=True)
charts_dir = ROOT.parents[1] / "05_evidencias" / "graficas"
charts_dir.mkdir(parents=True, exist_ok=True)
res.to_csv(models_dir / "benchmark_anomaly.csv", index=False)
sens.to_csv(models_dir / "benchmark_anomaly_sensitivity.csv", index=False)

display(res)
display(sens.head(15))


In [ ]:
plt.figure(figsize=(9, 4))
plt.bar(res["algoritmo"], res["score_general"])
plt.title("Score general por algoritmo")
plt.ylabel("score_general")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.savefig(charts_dir / "anomaly_score_por_algoritmo.png", dpi=150)
plt.show()

for algo, sub in sens.groupby("algoritmo"):
    tmp = sub.sort_values("contamination")
    plt.plot(tmp["contamination"], tmp["score_general"], marker="o", label=algo)
plt.title("Sensibilidad: contamination vs score_general")
plt.xlabel("contamination")
plt.ylabel("score_general")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(charts_dir / "anomaly_sensibilidad_contamination.png", dpi=150)
plt.show()

display(Markdown(f"Graficas guardadas en: `{charts_dir}`"))


In [ ]:
best = res.sort_values("score_general", ascending=False).iloc[0]

if float(best["bootstrap_jaccard_top_anomalias"]) >= 0.6 and float(best["desviacion_target_contamination"]) <= 0.02:
    semaforo = "VERDE"
    estado = "detector estable y alineado al nivel de alertas objetivo"
elif float(best["bootstrap_jaccard_top_anomalias"]) >= 0.35 and float(best["desviacion_target_contamination"]) <= 0.05:
    semaforo = "AMARILLO"
    estado = "detector util, pero requiere validacion manual recurrente"
else:
    semaforo = "ROJO"
    estado = "detector inestable o con volumen de alertas no controlado"

display(
    Markdown(
        f'''
## Interpretacion
- Mejor algoritmo en este corte: **{best['algoritmo']}**.
- `score_general`: **{best['score_general']:.4f}**.
- `% anomalias detectadas`: **{best['pct_anomalias']:.4f}**.
- Estabilidad bootstrap (Jaccard top anomalias): **{best['bootstrap_jaccard_top_anomalias']:.4f}**.

Lectura recomendada:
1. Validar manualmente las anomalias top en negocio antes de accionar.
2. Comparar estabilidad mensual para detectar deriva.
3. Recordar que con pocas agencias la varianza del ranking puede subir.

## Conclusion ejecutiva
- Semaforo: **{semaforo}**.
- Estado: **{estado}**.
'''
    )
)
